In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("/home/mateuschinelatto/Experiments/ssvep-bci-nn/cross-subject")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmark_dataset import (
    load_data_from_users,
    load_freq_phase
)

DATASET_ROOT_PATH = Path("/home/mateuschinelatto/Experiments/data/benchmark/")

## Data Loading

In [2]:
# Configuration parameters
users = list(range(1, 36))
users_to_run = users.copy()

# Electrode indices for the electrodes Pz, PO5, PO3, POz, PO4, PO6, O1, Oz, O2
occipital_electrodes = np.array([47, 53, 54, 55, 56, 57, 60, 61, 62])
# occipital_electrodes = [61]

# Which frequencies to analyze
all_frequencies, phases = load_freq_phase(DATASET_ROOT_PATH.joinpath("Freq_Phase.mat"))
print("All frequencies in the dataset:", all_frequencies)
bci_frequencies = all_frequencies[:8]
indices = [np.where(all_frequencies == freq)[0][0] for freq in bci_frequencies]

# Dataset visual delay and sampling frequency
visual_delay = 160 # ms
sample_rate = 250 # Hz

# Bandpass filter configuration
freq_cut_low = 6
freq_cut_high = 50
filter_order = 10

# Optional CAR configuration on loaded data
apply_car = True
car_reference_channels = occipital_electrodes
car_target_channels = occipital_electrodes

print("Users of interest:", users)
print("Users to run:", users_to_run)
print("BCI frequencies:", bci_frequencies)
print("Indices of frequencies of interest:", indices)

All frequencies in the dataset: [ 8.   9.  10.  11.  12.  13.  14.  15.   8.2  9.2 10.2 11.2 12.2 13.2
 14.2 15.2  8.4  9.4 10.4 11.4 12.4 13.4 14.4 15.4  8.6  9.6 10.6 11.6
 12.6 13.6 14.6 15.6  8.8  9.8 10.8 11.8 12.8 13.8 14.8 15.8]
Users of interest: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
Users to run: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
BCI frequencies: [ 8.  9. 10. 11. 12. 13. 14. 15.]
Indices of frequencies of interest: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]


In [ ]:
all_data_CAR_norm = load_data_from_users(
    dataset_path="/home/mateuschinelatto/Experiments/data/benchmark/",
    users=users,
    visual_delay=visual_delay,
    filter_bandpass=True,
    apply_car=True,
    car_reference_channels=car_reference_channels,
    car_target_channels=car_target_channels,
    sample_rate=sample_rate,
    freq_cut_low=freq_cut_low,
    freq_cut_high=freq_cut_high,
    filter_order=filter_order,
    normalize=True
)

## CCA analysis based on Templates

To estimate subject similarity without leakage, we use the full 5 s trials and compute a leave-one-trial-out template for each user and frequency. For each receiver user, every held-out trial is compared with CCA against every donor user's template at the same stimulus frequency.

The resulting similarity matrix is directional:
- rows: receiver users whose trials are being tested
- columns: donor users whose templates are used

We also build a symmetrized version of the matrix to make clustering and nearest-neighbor inspection easier.

In [ ]:
# Use the full 5 s trials here.
# The 1 s windows are not independent and would leak trial-specific structure.

from cca import CCA
import pandas as pd

template_data = all_data_CAR_norm
analysis_users = users
analysis_channels = occipital_electrodes
analysis_freq_indices = indices
analysis_freqs = bci_frequencies


def cca_corr(eeg_a, eeg_b):
    try:
        _, _, corr = CCA(eeg_a, eeg_b)
        corr = float(np.real_if_close(corr))
    except np.linalg.LinAlgError:
        corr = np.nan
    return float(np.clip(corr, 0.0, 1.0)) if np.isfinite(corr) else np.nan

def build_leave_one_out_templates(freq_trials):
    num_trials = freq_trials.shape[-1]

    loo_templates = np.zeros(
        (num_trials, freq_trials.shape[0], freq_trials.shape[1]),
        dtype=np.float64,
    )

    for trial_idx in range(num_trials):
        keep_mask = np.ones(num_trials, dtype=bool)
        keep_mask[trial_idx] = False
        loo_templates[trial_idx] = freq_trials[:, :, keep_mask].mean(axis=-1)

    return loo_templates

print(
    "Per-user trial block shape:",
    template_data[0][analysis_channels, :, analysis_freq_indices[0], :].shape,
)

In [ ]:
num_users = len(analysis_users)
num_freqs = len(analysis_freq_indices)

similarity_by_freq = np.zeros((num_freqs, num_users, num_users), dtype=np.float64)

for freq_pos, freq_idx in enumerate(analysis_freq_indices):
    donor_trials_by_user = []
    donor_templates_by_user = []

    for user_pos in range(num_users):
        freq_trials = template_data[user_pos][analysis_channels, :, freq_idx, :]
        donor_trials_by_user.append(freq_trials)
        donor_templates_by_user.append(freq_trials.mean(axis=-1))

    for receiver_pos in range(num_users):
        receiver_trials = donor_trials_by_user[receiver_pos]
        loo_templates = build_leave_one_out_templates(receiver_trials)

        for donor_pos in range(num_users):
            trial_scores = []

            for trial_idx in range(receiver_trials.shape[-1]):
                test_trial = receiver_trials[:, :, trial_idx]

                if donor_pos == receiver_pos:
                    donor_template = loo_templates[trial_idx]
                else:
                    donor_template = donor_templates_by_user[donor_pos]

                trial_scores.append(cca_corr(test_trial, donor_template))

            similarity_by_freq[freq_pos, receiver_pos, donor_pos] = np.nanmean(trial_scores)

similarity_by_freq_z = np.zeros_like(similarity_by_freq)
for freq_pos in range(num_freqs):
    freq_matrix = similarity_by_freq[freq_pos]
    similarity_by_freq_z[freq_pos] = (freq_matrix - freq_matrix.mean()) / (
        freq_matrix.std() + 1e-8
    )

similarity_mean_raw = similarity_by_freq.mean(axis=0)
similarity_mean_z = similarity_by_freq_z.mean(axis=0)

# Symmetric version is easier to interpret as "user-to-user similarity"
similarity_sym_raw = 0.5 * (similarity_mean_raw + similarity_mean_raw.T)
similarity_sym_z = 0.5 * (similarity_mean_z + similarity_mean_z.T)

print("Raw mean similarity shape:", similarity_mean_raw.shape)
print("Symmetric similarity shape:", similarity_sym_z.shape)

In [ ]:
user_labels = [f"S{user:02d}" for user in analysis_users]

fig, axes = plt.subplots(1, 3, figsize=(26, 7), dpi=180)

sns.heatmap(
    pd.DataFrame(similarity_mean_raw, index=user_labels, columns=user_labels),
    cmap="viridis",
    ax=axes[0],
)
axes[0].set_title("Mean raw CCA similarity\nrows = test user, cols = template user")
axes[0].set_xlabel("Template user")
axes[0].set_ylabel("Test user")

sns.heatmap(
    pd.DataFrame(similarity_mean_z, index=user_labels, columns=user_labels),
    cmap="vlag",
    center=0,
    ax=axes[1],
)
axes[1].set_title("Mean frequency-normalized similarity\nrows = test user, cols = template user")
axes[1].set_xlabel("Template user")
axes[1].set_ylabel("Test user")

sns.heatmap(
    pd.DataFrame(similarity_sym_z, index=user_labels, columns=user_labels),
    cmap="vlag",
    center=0,
    ax=axes[2],
)
axes[2].set_title("Symmetric similarity for clustering\n0.5 * (A→B + B→A)")
axes[2].set_xlabel("User")
axes[2].set_ylabel("User")

for ax in axes:
    ax.tick_params(axis="x", rotation=90)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 12), dpi=180)
axes = axes.ravel()

for freq_pos, freq in enumerate(analysis_freqs):
    sns.heatmap(
        pd.DataFrame(similarity_by_freq_z[freq_pos], index=user_labels, columns=user_labels),
        cmap="vlag",
        center=0,
        ax=axes[freq_pos],
        cbar=freq_pos == 0,
    )
    axes[freq_pos].set_title(f"{freq:.2f} Hz")
    axes[freq_pos].set_xlabel("Template user")
    axes[freq_pos].set_ylabel("Test user")
    axes[freq_pos].tick_params(axis="x", rotation=90)
    axes[freq_pos].tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

sns.clustermap(
    pd.DataFrame(similarity_sym_z, index=user_labels, columns=user_labels),
    cmap="vlag",
    center=0,
    linewidths=0.2,
    figsize=(12, 12),
)
plt.show()

In [ ]:
top_k = 5
ranking_matrix = similarity_sym_z.copy()
np.fill_diagonal(ranking_matrix, -np.inf)

rows = []
for user_pos, user in enumerate(analysis_users):
    donor_order = np.argsort(ranking_matrix[user_pos])[::-1][:top_k]
    for rank, donor_pos in enumerate(donor_order, start=1):
        rows.append(
            {
                "user": user,
                "rank": rank,
                "similar_user": analysis_users[donor_pos],
                "symmetric_z_similarity": similarity_sym_z[user_pos, donor_pos],
                "symmetric_raw_similarity": similarity_sym_raw[user_pos, donor_pos],
            }
        )

neighbors_df = pd.DataFrame(rows)
neighbors_df.head(25)